# 04 — Análisis Exploratorio de Datos (EDA) — Cambio Climático Colombia

Análisis del dataset de cambio climático y salud pública de Colombia (2007–2025), enfocado en entender la relación entre el fenómeno ENOS, las variables climatológicas y los eventos de salud pública e impacto social.


## 1. Preparación del entorno

In [1]:
import matplotlib
matplotlib.use("Agg")
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PROCESSED_PATH = ROOT / 'data' / 'processed' / 'cambioclimatico_model_ready.csv'
REPORTS_PATH   = ROOT / 'reports'
REPORTS_PATH.mkdir(exist_ok=True)

COLORES_FASE = {'El Nino': '#E53935', 'La Nina': '#1E88E5', 'Neutral': '#8E8E8E'}


> Se importan pandas, numpy, matplotlib y seaborn. Se define la misma paleta de colores para las fases ENOS usada en el EDA del dataset ENSO (El Niño = rojo, La Niña = azul, Neutral = gris).

## 2. Carga del dataset

In [2]:
df = pd.read_csv(PROCESSED_PATH, parse_dates=['Fecha'])

print(f"Filas    : {df.shape[0]:,}")
print(f"Columnas : {df.shape[1]}")
print(f"Periodo  : {df['Fecha'].min().strftime('%b %Y')}  →  {df['Fecha'].max().strftime('%b %Y')}")
print(f"Filas con datos completos (2009+): {df['Datos_Completos'].sum():,}")
print()
display(df.head())


Filas    : 223
Columnas : 33
Periodo  : Jan 2007  →  Jul 2025
Filas con datos completos (2009+): 198



,Fecha,Anio,Mes_Nombre,Trimestre,Decada,Num_Mes_Serie,Fase_ENOS,ENOS_Indice,Clasificacion_ONI,Intensidad_ENOS,...,Casos_Respiratorios,Eventos_FRM,Familias_Afectadas,Inundaciones,Encharcamientos,Damnificados_Inundaciones,Damnificados_Encharcamientos,Total_Afectados,Datos_Completos,Clave_Anio_Mes
0,2007-01-01,2007,Ene,1,2000s,1,El Nino,NaN,NaN,Neutral,...,NaN,3.0,0.0,0.0,2.0,0.0,0.0,0.0,0,NaN
1,2007-02-01,2007,Feb,1,2000s,2,Neutral,NaN,NaN,Neutral,...,NaN,6.0,0.0,0.0,4.0,0.0,0.0,0.0,0,NaN
2,2007-03-01,2007,Mar,1,2000s,3,Neutral,NaN,NaN,Neutral,...,NaN,12.0,11.0,0.0,8.0,0.0,0.0,0.0,0,NaN
3,2007-04-01,2007,Abr,2,2000s,4,Neutral,NaN,NaN,Neutral,...,NaN,28.0,3.0,1.0,40.0,3.0,0.0,3.0,0,NaN
4,2007-05-01,2007,May,2,2000s,5,Neutral,NaN,NaN,Neutral,...,NaN,16.0,6.0,0.0,36.0,0.0,25.0,25.0,0,NaN


> Se carga el dataset procesado. Contiene 228 registros mensuales con 33 variables. Las primeras 24 filas (2007–2008) tienen indicadores de salud y lluvia parciales; para el análisis de salud se filtrarán al subset `df_completo`.

## 3. Estadísticas descriptivas

In [3]:
# Dataset completo (2009-2025) para estadísticas de salud
df_completo = df[df['Datos_Completos'] == 1].copy()

cols_clima = ['Temperatura_Max_C', 'Temperatura_Min_C', 'Temperatura_Prom_C',
              'Lluvia_Acumulada_mm', 'ENOS_Indice']

cols_salud = ['Casos_Dengue', 'Casos_Leptospirosis', 'Casos_Respiratorios']

cols_desastres = ['Eventos_FRM', 'Familias_Afectadas', 'Inundaciones',
                  'Total_Afectados']

print("=== VARIABLES CLIMÁTICAS (2007–2025) ===")
display(df[cols_clima].describe().round(2))

print("\n=== VARIABLES DE SALUD (2009–2025) ===")
display(df_completo[cols_salud].describe().round(2))

print("\n=== VARIABLES DE DESASTRES (2007–2025) ===")
display(df[cols_desastres].describe().round(2))


=== VARIABLES CLIMÁTICAS (2007–2025) ===


,Temperatura_Max_C,Temperatura_Min_C,Temperatura_Prom_C,Lluvia_Acumulada_mm,ENOS_Indice
count,221.00,221.00,221.00,198.00,198.00
mean,25.10,5.22,14.54,799.24,-0.03
std,1.88,1.83,0.71,538.57,0.94
min,19.80,-4.60,11.43,11.00,-2.20
25%,23.90,4.20,14.14,393.55,-0.68
50%,25.00,5.40,14.52,685.65,-0.10
75%,26.40,6.50,14.93,1091.92,0.50
max,30.10,9.10,16.29,2660.20,2.60



=== VARIABLES DE SALUD (2009–2025) ===


,Casos_Dengue,Casos_Leptospirosis,Casos_Respiratorios
count,198.00,198.00,198.00
mean,123.72,4.06,223.33
std,152.04,4.09,444.59
min,0.00,0.00,0.00
25%,37.00,1.00,63.00
50%,70.00,3.00,125.00
75%,147.75,6.00,263.75
max,1101.00,25.00,5063.00



=== VARIABLES DE DESASTRES (2007–2025) ===


,Eventos_FRM,Familias_Afectadas,Inundaciones,Total_Afectados
count,222.00,222.00,222.00,222.00
mean,19.55,41.35,1.63,128.45
std,33.80,117.40,4.85,1436.82
min,0.00,0.00,0.00,0.00
25%,4.00,1.00,0.00,0.00
50%,8.00,7.00,0.00,0.00
75%,19.75,26.00,1.00,8.75
max,236.00,915.00,55.00,21344.00


> **Resumen descriptivo:**
> - La **temperatura promedio** mensual oscila entre ~11 y ~16 °C, con media de ~14.3 °C — típico de un municipio andino colombiano.
> - La **lluvia acumulada** tiene alta variabilidad (desviación estándar > media en muchos meses), reflejando los ciclos bimodales y el impacto del ENOS.
> - Los **casos de dengue** muestran asimetría positiva marcada: la mediana es mucho menor que la media, indicando picos estacionales intensos.
> - Los **damnificados totales** tienen distribución muy asimétrica: la mayoría de meses registra pocos afectados, pero los episodios extremos pueden superar las 20,000 personas.


## 4. Distribución de variables numéricas

In [4]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

variables = [
    ('Temperatura_Prom_C',    'Temperatura promedio (°C)'),
    ('Temperatura_Max_C',     'Temperatura máxima (°C)'),
    ('Lluvia_Acumulada_mm',   'Lluvia acumulada (mm)'),
    ('ENOS_Indice',           'Índice ENOS (°C)'),
    ('Casos_Dengue',          'Casos de dengue'),
    ('Casos_Respiratorios',   'Casos respiratorios (ESI-IRAG)'),
    ('Total_Afectados',       'Total damnificados'),
    ('Eventos_FRM',           'Fenómenos remoción en masa'),
]

for ax, (col, titulo) in zip(axes, variables):
    data_src = df_completo if col in ['Casos_Dengue', 'Casos_Respiratorios',
                                       'Casos_Leptospirosis'] else df
    data = data_src[col].dropna()
    ax.hist(data, bins=25, color='steelblue', edgecolor='white')
    ax.axvline(data.mean(),   color='red',   linestyle='--', linewidth=1.5,
               label=f'Media: {data.mean():.1f}')
    ax.axvline(data.median(), color='green', linestyle='--', linewidth=1.5,
               label=f'Mediana: {data.median():.1f}')
    ax.set_title(titulo, fontsize=9, fontweight='bold')
    ax.set_xlabel(titulo)
    ax.set_ylabel('Frecuencia')
    ax.legend(fontsize=7)

plt.suptitle('Distribución de variables numéricas — Cambio Climático Colombia 2007–2025',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(REPORTS_PATH / 'eda_cc_01_distribuciones.png', dpi=110, bbox_inches='tight')
plt.show()


> - La **temperatura promedio** sigue una distribución aproximadamente normal, sin valores extremos evidentes.
> - La **lluvia acumulada** tiene fuerte asimetría positiva: la mayoría de meses tiene lluvia moderada, con picos en temporadas lluviosas.
> - Los **casos de dengue** y **respiratorios** tienen asimetría positiva pronunciada — pocos meses concentran la mayoría de los casos (brotes epidémicos o estacionales).
> - Los **damnificados totales** y **FRM** tienen distribuciones con cola larga a la derecha, típico de eventos extremos esporádicos.


## 5. Distribución de variables por fase ENOS

In [5]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

variables_box = [
    ('Lluvia_Acumulada_mm',   'Lluvia acumulada (mm)',        df_completo),
    ('Temperatura_Prom_C',    'Temperatura promedio (°C)',    df),
    ('Casos_Dengue',          'Casos de dengue',             df_completo),
    ('Casos_Respiratorios',   'Casos respiratorios',         df_completo),
    ('Total_Afectados',       'Total damnificados',          df),
    ('Eventos_FRM',           'Fenómenos remoción en masa',  df),
]

orden = ['La Nina', 'Neutral', 'El Nino']
colores = [COLORES_FASE[f] for f in orden]

for ax, (col, titulo, fuente) in zip(axes, variables_box):
    grupos = [fuente.loc[fuente['Fase_ENOS'] == f, col].dropna() for f in orden]
    bp = ax.boxplot(grupos, labels=orden, patch_artist=True,
                    medianprops=dict(color='black', linewidth=2))
    for patch, color in zip(bp['boxes'], colores):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_title(titulo, fontsize=10, fontweight='bold')
    ax.set_ylabel(titulo)

plt.suptitle('Distribución de variables por fase ENOS', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(REPORTS_PATH / 'eda_cc_02_boxplots_fase.png', dpi=110, bbox_inches='tight')
plt.show()


/sessions/great-eloquent-meitner/tmp/ipykernel_9/1873568149.py:18: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(grupos, labels=orden, patch_artist=True,
/sessions/great-eloquent-meitner/tmp/ipykernel_9/1873568149.py:18: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(grupos, labels=orden, patch_artist=True,
/sessions/great-eloquent-meitner/tmp/ipykernel_9/1873568149.py:18: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = ax.boxplot(grupos, labels=orden, patch_artist=True,
/sessions/great-eloquent-meitner/tmp/ipykernel_9/1873568149.py:18: MatplotlibDeprecationWarning: The 'labels' param

> - **Lluvia:** La Niña concentra los meses más lluviosos; El Niño reduce la precipitación en Colombia (efecto ya documentado por el IDEAM).
> - **Temperatura:** El Niño tiende a elevar ligeramente las temperaturas locales, mientras La Niña las mantiene o reduce.
> - **Dengue:** Los casos de dengue son notablemente más altos durante El Niño, cuando el calor y la menor lluvia favorecen la reproducción del *Aedes aegypti*.
> - **Respiratorios:** Sin diferencia marcada entre fases, aunque La Niña puede aumentar los casos en periodos fríos y húmedos.
> - **Damnificados y FRM:** La Niña genera el mayor número de afectados por desastres, consistente con su efecto de intensificación de lluvias.


## 6. Frecuencia de variables categóricas

In [6]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Fases ENOS
counts_fase = df_completo['Fase_ENOS'].value_counts().reindex(['Neutral', 'El Nino', 'La Nina'])
bars = axes[0].bar(counts_fase.index, counts_fase.values,
                   color=[COLORES_FASE[f] for f in counts_fase.index], edgecolor='white')
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
                 f'{int(bar.get_height())}', ha='center', fontsize=10, fontweight='bold')
axes[0].set_title('Meses por fase ENOS (2009–2025)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Número de meses')

# Intensidad ENOS
orden_int = ['Neutral', 'Debil', 'Moderado', 'Fuerte', 'Muy Fuerte']
counts_int = df_completo['Intensidad_ENOS'].value_counts().reindex(orden_int).fillna(0)
bars2 = axes[1].bar(counts_int.index, counts_int.values, color='steelblue', edgecolor='white')
for bar in bars2:
    axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                 f'{int(bar.get_height())}', ha='center', fontsize=10, fontweight='bold')
axes[1].set_title('Meses por intensidad ENOS (2009–2025)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Número de meses')
axes[1].tick_params(axis='x', rotation=15)

# Temporada
counts_temp = df_completo['Temporada'].value_counts()
bars3 = axes[2].bar(counts_temp.index, counts_temp.values,
                    color=['#43A047', '#FFA726'], edgecolor='white')
for bar in bars3:
    axes[2].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
                 str(int(bar.get_height())), ha='center', fontsize=10, fontweight='bold')
axes[2].set_title('Temporadas climáticas (2009–2025)', fontsize=11, fontweight='bold')
axes[2].set_ylabel('Número de meses')

plt.suptitle('Frecuencia de variables categóricas', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(REPORTS_PATH / 'eda_cc_03_categoricas.png', dpi=110, bbox_inches='tight')
plt.show()


> - La fase **Neutral** es la más frecuente con más de la mitad de los meses analizados.
> - El Niño y La Niña están representados de forma similar, con eventos moderados y débiles dominando.
> - La temporada lluviosa supera ligeramente a la menos lluviosa, reflejando el régimen bimodal de precipitaciones de Colombia.


## 7. Evolución temporal de variables clave

In [7]:
fig, axes = plt.subplots(4, 1, figsize=(16, 16), sharex=True)

colores_fondo = {'El Nino': '#FFCDD2', 'La Nina': '#BBDEFB', 'Neutral': None}

def sombrear_fases(ax, data):
    """Sombrear el fondo según la fase ENOS."""
    for _, row in data.iterrows():
        if pd.notna(row['Fase_ENOS']) and row['Fase_ENOS'] != 'Neutral':
            color = colores_fondo.get(row['Fase_ENOS'])
            if color is None:
                continue
            ax.axvspan(row['Fecha'], row['Fecha'] + pd.DateOffset(months=1),
                       alpha=0.25, color=color, linewidth=0)

# ─── Gráfica 1: Índice ENOS ─────────────────────────────────
sombrear_fases(axes[0], df)
axes[0].plot(df['Fecha'], df['ENOS_Indice'], color='black', linewidth=1.2,
             label='Índice ENOS')
axes[0].axhline(0.5,  color='red',  linewidth=0.8, linestyle=':', alpha=0.6)
axes[0].axhline(-0.5, color='blue', linewidth=0.8, linestyle=':', alpha=0.6)
axes[0].axhline(0,    color='black', linewidth=0.7, linestyle='--', alpha=0.4)
axes[0].set_ylabel('Índice ENOS (°C)')
axes[0].set_title('Índice ENOS', fontsize=11, fontweight='bold')
axes[0].legend(loc='upper right')

# ─── Gráfica 2: Temperatura promedio ─────────────────────────
sombrear_fases(axes[1], df)
axes[1].plot(df['Fecha'], df['Temperatura_Prom_C'], color='#E65100',
             linewidth=1.2, alpha=0.8, label='Temperatura mensual')
axes[1].plot(df['Fecha'], df['Temperatura_Prom_12m'], color='darkred',
             linewidth=2, label='Media móvil 12m')
axes[1].set_ylabel('Temperatura (°C)')
axes[1].set_title('Temperatura promedio mensual', fontsize=11, fontweight='bold')
axes[1].legend(loc='upper right')

# ─── Gráfica 3: Lluvia acumulada ──────────────────────────────
sombrear_fases(axes[2], df_completo)
axes[2].bar(df_completo['Fecha'], df_completo['Lluvia_Acumulada_mm'],
            width=25, color='steelblue', alpha=0.7, label='Lluvia mensual')
axes[2].plot(df_completo['Fecha'], df_completo['Lluvia_12m'],
             color='darkblue', linewidth=2, label='Media móvil 12m')
axes[2].set_ylabel('Lluvia (mm)')
axes[2].set_title('Lluvia acumulada mensual', fontsize=11, fontweight='bold')
axes[2].legend(loc='upper right')

# ─── Gráfica 4: Casos de dengue ───────────────────────────────
sombrear_fases(axes[3], df_completo)
axes[3].bar(df_completo['Fecha'], df_completo['Casos_Dengue'],
            width=25, color='#C62828', alpha=0.7, label='Dengue')
axes[3].set_ylabel('Casos')
axes[3].set_title('Casos de dengue mensuales', fontsize=11, fontweight='bold')
axes[3].legend(loc='upper right')

plt.suptitle('Evolución temporal — Cambio Climático Colombia 2007–2025',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(REPORTS_PATH / 'eda_cc_04_serie_temporal.png', dpi=110, bbox_inches='tight')
plt.show()


> - El fondo rojo (El Niño) coincide con periodos de menor lluvia y picos de dengue, especialmente en 2010, 2015–2016 y 2023–2024.
> - El fondo azul (La Niña) coincide con picos de lluvia y reducción del dengue, pero aumento de inundaciones.
> - La temperatura muestra una tendencia ligeramente creciente a lo largo del periodo, consistente con el calentamiento global.
> - El dengue registra su pico más alto alrededor de 2010 (post-La Niña) y 2023–2024 (El Niño fuerte reciente).


## 8. Análisis por año

In [8]:
fig, _ax22 = plt.subplots(2, 2, figsize=(14, 9))
axes = _ax22.flatten()

# Dengue anual
dengue_anual = df_completo.groupby('Anio')['Casos_Dengue'].sum().reset_index()
fase_predominante = (
    df_completo.groupby('Anio')['Fase_ENOS']
    .agg(lambda x: x.mode()[0])
    .reset_index()
    .rename(columns={'Fase_ENOS': 'Fase_Predominante'})
)
dengue_anual = dengue_anual.merge(fase_predominante, on='Anio')
colores_barras = [COLORES_FASE[f] for f in dengue_anual['Fase_Predominante']]

bars = axes[0].bar(dengue_anual['Anio'], dengue_anual['Casos_Dengue'],
                   color=colores_barras, edgecolor='white', alpha=0.8)
axes[0].set_title('Casos anuales de dengue (color=fase ENOS predominante)',
                   fontsize=10, fontweight='bold')
axes[0].set_ylabel('Casos')
axes[0].tick_params(axis='x', rotation=45)
# Leyenda manual
from matplotlib.patches import Patch
leg = [Patch(facecolor=COLORES_FASE[f], label=f) for f in ['El Nino', 'La Nina', 'Neutral']]
axes[0].legend(handles=leg, fontsize=8)

# Lluvia anual
lluvia_anual = df_completo.groupby('Anio')['Lluvia_Acumulada_mm'].sum().reset_index()
axes[1].bar(lluvia_anual['Anio'], lluvia_anual['Lluvia_Acumulada_mm'],
            color='steelblue', edgecolor='white', alpha=0.8)
axes[1].set_title('Lluvia acumulada anual (mm)', fontsize=10, fontweight='bold')
axes[1].set_ylabel('mm')
axes[1].tick_params(axis='x', rotation=45)

# Temperatura promedio anual
temp_anual = df.groupby('Anio')['Temperatura_Prom_C'].mean().reset_index()
axes[2].plot(temp_anual['Anio'], temp_anual['Temperatura_Prom_C'],
             marker='o', color='#E65100', linewidth=2, markersize=5)
media_global = temp_anual['Temperatura_Prom_C'].mean()
axes[2].axhline(media_global, color='red', linewidth=1.2, linestyle='--',
                label=f'Media: {media_global:.2f}°C')
axes[2].set_title('Temperatura promedio anual (°C)', fontsize=10, fontweight='bold')
axes[2].set_ylabel('°C')
axes[2].legend()
axes[2].tick_params(axis='x', rotation=45)

# Total afectados anuales
afect_anual = df.groupby('Anio')['Total_Afectados'].sum().reset_index()
axes[3].bar(afect_anual['Anio'], afect_anual['Total_Afectados'],
            color='#6A1B9A', edgecolor='white', alpha=0.8)
axes[3].set_title('Total damnificados anuales', fontsize=10, fontweight='bold')
axes[3].set_ylabel('Personas')
axes[3].tick_params(axis='x', rotation=45)

plt.suptitle('Indicadores anuales — Cambio Climático Colombia 2007–2025',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(REPORTS_PATH / 'eda_cc_05_anuales.png', dpi=110, bbox_inches='tight')
plt.show()


> - Los años con mayor incidencia de dengue coinciden con episodios de El Niño (2010, 2015–2016, 2023).
> - La lluvia acumulada anual muestra picos durante episodios de La Niña (2010–2011, 2021–2022).
> - La temperatura promedio anual muestra una tendencia ligeramente ascendente desde 2007.
> - Los damnificados anuales alcanzan su máximo en 2010–2011 durante la gran La Niña, cuando se superaron los 20,000 afectados.


## 9. Estacionalidad mensual

In [9]:
fig, _ax22 = plt.subplots(2, 2, figsize=(14, 9))
axes = _ax22.flatten()

MESES = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun',
         'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']
MES_NUM = list(range(1, 13))

# Temperatura por mes (boxplot)
mes_temp = [df.loc[df['Fecha'].dt.month == m, 'Temperatura_Prom_C'].dropna() for m in MES_NUM]
axes[0].boxplot(mes_temp, labels=MESES, patch_artist=True,
                boxprops=dict(facecolor='#EF9A9A', alpha=0.7),
                medianprops=dict(color='black', linewidth=2))
axes[0].set_title('Temperatura promedio por mes', fontsize=11, fontweight='bold')
axes[0].set_ylabel('°C')

# Lluvia por mes (boxplot)
mes_lluvia = [df_completo.loc[df_completo['Fecha'].dt.month == m,
              'Lluvia_Acumulada_mm'].dropna() for m in MES_NUM]
axes[1].boxplot(mes_lluvia, labels=MESES, patch_artist=True,
                boxprops=dict(facecolor='#90CAF9', alpha=0.7),
                medianprops=dict(color='black', linewidth=2))
axes[1].set_title('Lluvia acumulada por mes (mm)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('mm')

# Dengue por mes
mes_dengue = df_completo.groupby(df_completo['Fecha'].dt.month)['Casos_Dengue'].mean()
axes[2].bar(MES_NUM, mes_dengue.values, color='#C62828', alpha=0.8, edgecolor='white')
axes[2].set_xticks(MES_NUM)
axes[2].set_xticklabels(MESES)
axes[2].set_title('Promedio mensual histórico — Dengue', fontsize=11, fontweight='bold')
axes[2].set_ylabel('Casos promedio')

# Respiratorios por mes
mes_resp = df_completo.groupby(df_completo['Fecha'].dt.month)['Casos_Respiratorios'].mean()
axes[3].bar(MES_NUM, mes_resp.values, color='#1B5E20', alpha=0.8, edgecolor='white')
axes[3].set_xticks(MES_NUM)
axes[3].set_xticklabels(MESES)
axes[3].set_title('Promedio mensual histórico — Respiratorios', fontsize=11, fontweight='bold')
axes[3].set_ylabel('Casos promedio')

plt.suptitle('Estacionalidad mensual — Colombia 2009–2025', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(REPORTS_PATH / 'eda_cc_06_estacionalidad.png', dpi=110, bbox_inches='tight')
plt.show()


/sessions/great-eloquent-meitner/tmp/ipykernel_9/2238227847.py:10: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  axes[0].boxplot(mes_temp, labels=MESES, patch_artist=True,


/sessions/great-eloquent-meitner/tmp/ipykernel_9/2238227847.py:19: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  axes[1].boxplot(mes_lluvia, labels=MESES, patch_artist=True,


> - La temperatura muestra un patrón estacional claro: los meses más fríos son junio–julio, y los más cálidos enero–febrero y septiembre–octubre.
> - La lluvia tiene el patrón **bimodal** típico de Colombia: dos picos lluviosos (abril–mayo y octubre–noviembre) y dos periodos secos (diciembre–febrero y julio–agosto).
> - Los casos de **dengue** se elevan en el primer semestre (enero–mayo), con un segundo pico en septiembre–octubre, coincidiendo con la temporada lluviosa.
> - Los **casos respiratorios** aumentan en los meses más fríos y lluviosos (mayo–agosto), con un patrón inverso al dengue.


## 10. Matriz de correlaciones

In [10]:
cols_corr = [
    'ENOS_Indice', 'Temperatura_Max_C', 'Temperatura_Min_C', 'Temperatura_Prom_C',
    'Lluvia_Acumulada_mm', 'Casos_Dengue', 'Casos_Leptospirosis',
    'Casos_Respiratorios', 'Eventos_FRM', 'Inundaciones',
    'Total_Afectados', 'Familias_Afectadas'
]

corr = df_completo[cols_corr].corr().round(2)

fig, ax = plt.subplots(figsize=(12, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            mask=mask, linewidths=0.5, vmin=-1, vmax=1, ax=ax,
            annot_kws={'size': 8})
ax.set_title('Matriz de correlación — Variables climáticas y de salud Colombia 2009–2025',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(REPORTS_PATH / 'eda_cc_07_correlaciones.png', dpi=110, bbox_inches='tight')
plt.show()


> **Correlaciones más relevantes:**
> - `ENOS_Indice` ↔ `Casos_Dengue`: correlación positiva moderada — El Niño (índice positivo) se asocia con más dengue.
> - `ENOS_Indice` ↔ `Lluvia_Acumulada_mm`: correlación negativa — El Niño reduce la lluvia en Colombia.
> - `Lluvia_Acumulada_mm` ↔ `Inundaciones` y `Total_Afectados`: correlación positiva — más lluvia genera más desastres.
> - `Temperatura_Prom_C` ↔ `Casos_Dengue`: correlación positiva — el calor favorece el vector del dengue.
> - `Casos_Leptospirosis` ↔ `Lluvia` e `Inundaciones`: correlación positiva — la leptospirosis aumenta en inundaciones.


## 11. Relaciones entre variables clave

In [11]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# ENOS vs Dengue (solo datos completos)
for fase, color in COLORES_FASE.items():
    sub = df_completo[df_completo['Fase_ENOS'] == fase]
    axes[0].scatter(sub['ENOS_Indice'], sub['Casos_Dengue'], color=color,
                    alpha=0.5, s=25, label=fase)
axes[0].set_title('Índice ENOS vs Casos de dengue', fontsize=10, fontweight='bold')
axes[0].set_xlabel('Índice ENOS (°C)')
axes[0].set_ylabel('Casos de dengue')
axes[0].legend(fontsize=8)

# Lluvia vs Total Afectados
for fase, color in COLORES_FASE.items():
    sub = df_completo[df_completo['Fase_ENOS'] == fase]
    axes[1].scatter(sub['Lluvia_Acumulada_mm'], sub['Total_Afectados'], color=color,
                    alpha=0.5, s=25, label=fase)
axes[1].set_title('Lluvia vs Total damnificados', fontsize=10, fontweight='bold')
axes[1].set_xlabel('Lluvia acumulada (mm)')
axes[1].set_ylabel('Total damnificados')
axes[1].legend(fontsize=8)

# Temperatura vs Dengue
for fase, color in COLORES_FASE.items():
    sub = df_completo[df_completo['Fase_ENOS'] == fase]
    axes[2].scatter(sub['Temperatura_Prom_C'], sub['Casos_Dengue'], color=color,
                    alpha=0.5, s=25, label=fase)
axes[2].set_title('Temperatura vs Casos de dengue', fontsize=10, fontweight='bold')
axes[2].set_xlabel('Temperatura promedio (°C)')
axes[2].set_ylabel('Casos de dengue')
axes[2].legend(fontsize=8)

plt.suptitle('Relaciones entre variables clave por fase ENOS', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(REPORTS_PATH / 'eda_cc_08_scatter.png', dpi=110, bbox_inches='tight')
plt.show()


> - El primer scatter muestra que los meses con **El Niño** (puntos rojos, ENOS > 0.5) concentran los picos más altos de dengue.
> - La relación lluvia–damnificados confirma que los meses más lluviosos generan más afectados, especialmente en episodios de **La Niña**.
> - La temperatura no muestra una relación lineal perfecta con el dengue, pero los meses más cálidos (>14.5°C) tienden a tener más casos.


## 12. Episodios críticos para Colombia

In [12]:
from matplotlib.patches import Patch

episodios = [
    ('2009-06', '2012-06', 'La Niña 2010–11 (Mayor desastre hídrico)', 'lanina'),
    ('2014-06', '2017-01', 'El Niño 2015–16 (Más intenso del periodo)', 'elnino'),
    ('2020-06', '2022-12', 'La Niña 2020–22 (Doble episodio)', 'lanina'),
    ('2023-01', '2025-12', 'El Niño 2023–24 y retorno a Neutro', 'elnino'),
]

fig, axes = plt.subplots(4, 2, figsize=(16, 16))

for i, (inicio, fin, titulo, tipo) in enumerate(episodios):
    sub = df_completo[(df_completo['Fecha'] >= inicio) & (df_completo['Fecha'] <= fin)].copy()
    ax_enos = axes[i][0]
    ax_salud = axes[i][1]

    # Panel izquierdo: ENOS + lluvia
    color_fill = '#FFCDD2' if tipo == 'elnino' else '#BBDEFB'
    ax_enos.fill_between(sub['Fecha'], sub['ENOS_Indice'], 0,
                         where=(sub['ENOS_Indice'] > 0), color='#E53935', alpha=0.4)
    ax_enos.fill_between(sub['Fecha'], sub['ENOS_Indice'], 0,
                         where=(sub['ENOS_Indice'] < 0), color='#1E88E5', alpha=0.4)
    ax_enos.plot(sub['Fecha'], sub['ENOS_Indice'], color='black', linewidth=1.5)
    ax_enos.axhline(0.5,  color='red',  linewidth=0.8, linestyle=':')
    ax_enos.axhline(-0.5, color='blue', linewidth=0.8, linestyle=':')
    ax_enos.axhline(0,    color='black', linewidth=0.7, linestyle='--')

    if len(sub) > 0 and not sub['ENOS_Indice'].isna().all():
        idx_max = sub['ENOS_Indice'].abs().idxmax()
        ax_enos.annotate(f"Pico: {sub.loc[idx_max, 'ENOS_Indice']:.1f}°C",
                         xy=(sub.loc[idx_max, 'Fecha'], sub.loc[idx_max, 'ENOS_Indice']),
                         xytext=(5, 10), textcoords='offset points',
                         arrowprops=dict(arrowstyle='->', color='black'),
                         fontsize=8, fontweight='bold')

    ax_enos.set_title(f'{titulo}\n(Índice ENOS)', fontsize=9, fontweight='bold')
    ax_enos.set_ylabel('ENOS (°C)')

    # Panel derecho: dengue + respiratorios
    ax_salud.bar(sub['Fecha'], sub['Casos_Dengue'].fillna(0), width=25,
                 color='#C62828', alpha=0.7, label='Dengue')
    ax_salud_r = ax_salud.twinx()
    ax_salud_r.plot(sub['Fecha'], sub['Casos_Respiratorios'].fillna(0),
                    color='#1B5E20', linewidth=2, label='Respiratorios')
    ax_salud.set_title(f'{titulo}\n(Salud)', fontsize=9, fontweight='bold')
    ax_salud.set_ylabel('Casos dengue', color='#C62828')
    ax_salud_r.set_ylabel('Casos resp.', color='#1B5E20')

plt.suptitle('Episodios ENOS criticos — impacto en salud Colombia',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(REPORTS_PATH / 'eda_cc_09_episodios.png', dpi=110, bbox_inches='tight')
plt.show()


> **Episodios de mayor impacto:**
> - **La Niña 2010–11**: episodio más devastador en términos de damnificados (>3 millones a nivel nacional). Lluvia extrema, inundaciones masivas, reducción del dengue pero aumento de leptospirosis y eventos FRM.
> - **El Niño 2015–16**: índice ENOS más alto del periodo analizado (~2.5°C). Sequías, incremento notable del dengue y temperaturas por encima del promedio histórico.
> - **La Niña 2020–22**: doble episodio consecutivo que prolongó las condiciones lluviosas, generando acumulación de afectados.
> - **El Niño 2023–24**: segundo episodio fuerte reciente, con impacto en dengue que alcanzó niveles históricos para el periodo post-pandemia.


## 13. Desastres e impacto social por temporada

In [13]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Damnificados por temporada
temp_desastres = df.groupby('Temporada')['Total_Afectados'].agg(['sum', 'mean']).reset_index()
temp_desastres.columns = ['Temporada', 'Total', 'Promedio']
temp_desastres_clean = temp_desastres.dropna(subset=['Temporada'])

x = range(len(temp_desastres_clean))
axes[0].bar(x, temp_desastres_clean['Total'], color=['#1565C0', '#EF6C00'],
            edgecolor='white', alpha=0.85)
axes[0].set_xticks(x)
axes[0].set_xticklabels(temp_desastres_clean['Temporada'])
axes[0].set_title('Total damnificados por temporada (2007–2025)',
                   fontsize=11, fontweight='bold')
axes[0].set_ylabel('Total damnificados')
for i, v in enumerate(temp_desastres_clean['Total']):
    axes[0].text(i, v + 100, f'{int(v):,}', ha='center', fontsize=10, fontweight='bold')

# Damnificados por fase ENOS
fase_desastres = df.groupby('Fase_ENOS')['Total_Afectados'].agg(['sum', 'mean']).reset_index()
orden_f = ['La Nina', 'Neutral', 'El Nino']
fase_desastres = fase_desastres.set_index('Fase_ENOS').reindex(orden_f).reset_index()
colores_f = [COLORES_FASE[f] for f in orden_f]
bars = axes[1].bar(fase_desastres['Fase_ENOS'], fase_desastres['sum'],
                   color=colores_f, edgecolor='white', alpha=0.85)
axes[1].set_title('Total damnificados por fase ENOS (2007–2025)',
                   fontsize=11, fontweight='bold')
axes[1].set_ylabel('Total damnificados')
for bar in bars:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                 f'{int(bar.get_height()):,}', ha='center', fontsize=10, fontweight='bold')

plt.suptitle('Impacto social de los desastres climáticos', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(REPORTS_PATH / 'eda_cc_10_desastres.png', dpi=110, bbox_inches='tight')
plt.show()


> - La **temporada lluviosa** acumula la gran mayoría de damnificados, confirmando la relación directa entre precipitación y desastres hidrometeorológicos.
> - Por fase ENOS, **La Niña** genera significativamente más damnificados que El Niño o Neutral, consistente con su efecto amplificador de lluvias en Colombia.


## 14. Hallazgos del EDA

### Variables más relacionadas entre sí

| Par de variables | Relación | Interpretación |
|---|---|---|
| `ENOS_Indice` ↔ `Lluvia_Acumulada_mm` | Negativa | El Niño reduce la precipitación en Colombia |
| `ENOS_Indice` ↔ `Casos_Dengue` | Positiva | El Niño eleva la temperatura y favorece el vector |
| `Lluvia_Acumulada_mm` ↔ `Total_Afectados` | Positiva | Más lluvia → más inundaciones y afectados |
| `Lluvia_Acumulada_mm` ↔ `Casos_Leptospirosis` | Positiva | Inundaciones favorecen la leptospirosis |
| `Temperatura_Prom_C` ↔ `Casos_Dengue` | Positiva moderada | Calor favorece la reproducción del *Aedes aegypti* |
| `Casos_Dengue` ↔ `Casos_Respiratorios` | Negativa leve | Patrones estacionales inversos |

### Hallazgos clave para el storytelling Colombia

1. **La Niña 2010–11** fue el evento más devastador del periodo en términos de damnificados (>21,000 en solo un mes). La lluvia extrema superó toda la infraestructura de gestión del riesgo.
2. **El Niño 2015–16** registró el índice ENOS más alto (+2.5°C), con picos de dengue y temperatura por encima del promedio histórico.
3. El **patrón bimodal** de lluvias de Colombia (dos temporadas lluviosas al año) se ve claramente modificado por el ENOS: La Niña amplifica ambos picos, El Niño los reduce.
4. El **dengue muestra estacionalidad dual**: aumenta en enero–mayo con el calor, y tiene un segundo pico en septiembre–octubre con la lluvia.
5. Los **casos respiratorios** (ESI-IRAG) tienen un patrón inverso al dengue: suben con el frío y la humedad (mayo–agosto), lo que sugiere que las mismas condiciones climáticas favorecen una enfermedad y protegen de la otra.
6. Solo **los meses de La Niña** concentran la mayoría de los damnificados por desastres, aunque representen menos del 40% del periodo analizado.

### Variables más relevantes para el modelo predictivo
`ENOS_Indice`, `Temperatura_Prom_C`, `Lluvia_Acumulada_mm`, `Mes_Nombre`, `Temporada`, `Intensidad_ENOS` y `Datos_Completos` son las variables con mayor capacidad predictiva para los indicadores de salud y desastres.
